# Coordinate Comparison — 2D/3D × Normalized/World

Tests whether MediaPipe **world coordinates** and/or the **3rd (depth) dimension**
improve joint-angle accuracy, for one out-of-plane task (hip internal rotation)
and one in-plane task (Nordic curl).

Four methods form a 2×2 grid:

|            | normalized file | world file |
|------------|-----------------|------------|
| **2D (x,y)**   | 2D-norm        | 2D-world   |
| **3D (x,y,z)** | 3D-norm        | 3D-world   |

Reading **across rows** isolates the effect of the *file* (world calibration);
reading **down columns** isolates the effect of adding the *depth dimension*.

Output: a long-format CSV (one row per task-subject-method) for analysis.

## Setup

In [1]:
import glob, os
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from utilities import utils
from utilities.utils import get_angle, smooth_array
from tasks import cmj, hip, nordic
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline

def _world_path(fp):
    """Given a normal keypoint file, return its _world twin path."""
    return fp.replace(".txt", "_world.txt")

def _peak_rom(signal):
    """Peak range of motion = max - min of the (finite) angle signal."""
    s = np.asarray(signal, dtype=float)
    s = s[np.isfinite(s)]
    if s.size < 2:
        return np.nan
    return float(np.nanmax(s) - np.nanmin(s))

## Angle helpers

For each method we pull the right columns (x,y for 2D; x,y,z for 3D) from the
right file (normalized or world) and compute the joint angle. We report **peak
ROM** per method and compare against the OMC ground-truth peak ROM.

In [2]:
def angle_from_file(fp, joints, ndims):
    """Compute a 3-point joint angle from a keypoint file.
    joints = (a, b, c) landmark names; b is the vertex. ndims=2 uses x,y;
    ndims=3 uses x,y,z."""
    df = cmj.load_pose_file(fp)
    pts = []
    for j in joints:
        xyz = df[[f"{j}_x", f"{j}_y", f"{j}_z"]].to_numpy().copy()
        pts.append(smooth_array(xyz, tuple(range(ndims))))
    a, b, c = pts
    return get_angle(a, b, c, ndims=ndims)

# --- hip uses a rotation angle (knee->ankle segment vs rest), not a 3-point angle
def hip_angle_from_file(fp, ndims):
    df = cmj.load_pose_file(fp)
    knee = df[["RIGHT_KNEE_x","RIGHT_KNEE_y","RIGHT_KNEE_z"]].to_numpy()[:, :ndims]
    ankle = df[["RIGHT_ANKLE_x","RIGHT_ANKLE_y","RIGHT_ANKLE_z"]].to_numpy()[:, :ndims]
    seg = ankle - knee
    rest = seg[0]
    cos = (seg @ rest) / (np.linalg.norm(seg, axis=1) * np.linalg.norm(rest) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

## Task configuration

Each task specifies how to find its files, its ground-truth angle, and how to
compute the MMC angle. Hip is front-camera and uses the rotation angle; Nordic is
side-camera and uses the hip–knee–ankle 3-point angle.

In [3]:
# ground-truth (OMC) peak ROM per subject, reused across all four methods
hip_data = utils.read_list('hip_data')
nordic_data = utils.read_list('nordic_data')

def hip_normal_file(sid):
    return hip._find_file("data/keypoints/hip", sid, "hir")

def nordic_normal_file(sid):
    cands = [p for p in glob.glob("data/keypoints/nordic/*.txt")
             if "world" not in p.lower() and sid in os.path.basename(p)]
    return cands[0] if cands else None

TASKS = {
    "Hip internal rotation": {
        "plane": "out-of-plane",
        "find_file": hip_normal_file,
        "omc": lambda a: hip._omc_angle(hip_data[a], "hir"),
        "mmc": lambda fp, ndims: hip_angle_from_file(fp, ndims),
        "exclude": {sid for (sid, t) in hip.EXCLUDE if t == "hir"},
    },
    "Nordic curl": {
        "plane": "in-plane (sagittal)",
        "find_file": nordic_normal_file,
        "omc": lambda a: nordic._omc_angle(nordic_data[a],
                     nordic.CODA_START.get(nordic.SUBJECT_IDS[a], nordic.CODA_START_DEFAULT)),
        "mmc": lambda fp, ndims: angle_from_file(
                     fp, ("RIGHT_HIP","RIGHT_KNEE","RIGHT_ANKLE"), ndims)[600:],
        "exclude": set(),
    },
}

## Build the comparison table

For every task × subject, compute the four methods (2D/3D × normalized/world) and
the OMC ground truth, then record peak ROM and error vs OMC for each method.

In [4]:
SUBJECT_IDS = [f"P{i:02d}" for i in range(3, 19)]

rows = []
for task_name, cfg in TASKS.items():
    for a, sid in enumerate(SUBJECT_IDS):
        if sid in cfg["exclude"]:
            continue
        normal_fp = cfg["find_file"](sid)
        if not normal_fp:
            continue
        world_fp = _world_path(normal_fp)
        if not os.path.exists(world_fp):
            print(f"[{task_name}] {sid}: no world file, skipped")
            continue
        try:
            omc_rom = _peak_rom(cfg["omc"](a))
            methods = {
                "2D_norm":  cfg["mmc"](normal_fp, 2),
                "3D_norm":  cfg["mmc"](normal_fp, 3),
                "2D_world": cfg["mmc"](world_fp, 2),
                "3D_world": cfg["mmc"](world_fp, 3),
            }
        except Exception as ex:
            print(f"[{task_name}] {sid}: {type(ex).__name__}: {str(ex)[:50]}")
            continue
        for method_name, signal in methods.items():
            rom = _peak_rom(signal)
            rows.append({
                "task": task_name,
                "plane": cfg["plane"],
                "subject": sid,
                "method": method_name,
                "coord_dims": "2D" if method_name.startswith("2D") else "3D",
                "coord_source": "world" if method_name.endswith("world") else "normalized",
                "mmc_rom": np.round(rom, 2),
                "omc_rom": np.round(omc_rom, 2),
                "error": np.round(abs(rom - omc_rom), 2),
            })

df = pd.DataFrame(rows)
print(df.shape, "rows")
df.head(8)

(124, 9) rows


,task,plane,subject,method,coord_dims,coord_source,mmc_rom,omc_rom,error
0,Hip internal rotation,out-of-plane,P03,2D_norm,2D,normalized,72.94,75.5,2.56
1,Hip internal rotation,out-of-plane,P03,3D_norm,3D,normalized,80.25,75.5,4.75
2,Hip internal rotation,out-of-plane,P03,2D_world,2D,world,57.33,75.5,18.17
3,Hip internal rotation,out-of-plane,P03,3D_world,3D,world,57.64,75.5,17.86
4,Hip internal rotation,out-of-plane,P04,2D_norm,2D,normalized,86.36,69.0,17.36
5,Hip internal rotation,out-of-plane,P04,3D_norm,3D,normalized,34.19,69.0,34.81
6,Hip internal rotation,out-of-plane,P04,2D_world,2D,world,65.45,69.0,3.55
7,Hip internal rotation,out-of-plane,P04,3D_world,3D,world,60.34,69.0,8.66


## Save to CSV

In [5]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/coordinate_comparison.csv", index=False)
print("saved results/coordinate_comparison.csv —", len(df), "rows")

saved results/coordinate_comparison.csv — 124 rows


## Analysis — mean error per method

Lower error = closer to the OMC ground truth. The pivot lays out the 2×2 grid so
you can read the effect of the file (across) and the depth dimension (down).

In [6]:
# mean error per method, per task
summary = (df.groupby(["task", "plane", "method"])["error"]
             .mean().round(2).reset_index()
             .rename(columns={"error": "mean_error"}))
summary

,task,plane,method,mean_error
0,Hip internal rotation,out-of-plane,2D_norm,10.01
1,Hip internal rotation,out-of-plane,2D_world,8.79
2,Hip internal rotation,out-of-plane,3D_norm,23.62
3,Hip internal rotation,out-of-plane,3D_world,10.75
4,Nordic curl,in-plane (sagittal),2D_norm,25.43
5,Nordic curl,in-plane (sagittal),2D_world,19.74
6,Nordic curl,in-plane (sagittal),3D_norm,33.70
7,Nordic curl,in-plane (sagittal),3D_world,19.22


In [7]:
# the 2x2 grid per task: rows = 2D/3D, cols = normalized/world
for task_name in df["task"].unique():
    sub = df[df["task"] == task_name]
    grid = sub.pivot_table(index="coord_dims", columns="coord_source",
                           values="error", aggfunc="mean").round(2)
    print(f"\n{task_name}  (mean error vs OMC, lower = better)")
    print(grid)


Hip internal rotation  (mean error vs OMC, lower = better)
coord_source  normalized  world
coord_dims                     
2D                 10.01   8.79
3D                 23.62  10.75

Nordic curl  (mean error vs OMC, lower = better)
coord_source  normalized  world
coord_dims                     
2D                 25.43  19.74
3D                 33.70  19.22


In [8]:
# quick verdict per task
for task_name in df["task"].unique():
    sub = df[df["task"] == task_name]
    best = sub.groupby("method")["error"].mean().idxmin()
    best_err = sub.groupby("method")["error"].mean().min()
    cur = sub[sub["method"] == "2D_norm"]["error"].mean()
    print(f"{task_name}: current (2D_norm) = {cur:.2f}; best = {best} ({best_err:.2f})")

Hip internal rotation: current (2D_norm) = 10.01; best = 2D_world (8.79)
Nordic curl: current (2D_norm) = 25.43; best = 3D_world (19.22)
